# TaaS Studio — Hypergraph Explorer hardening validation

This notebook validates the integrated Studio page and authenticated Discovery API using the real TaaS DiscoveryEngine with a local synthetic tool contract. It also checks degenerate LSA retrieval behavior and the Core projection/UI regressions. The synthetic tool is never invoked; this is not external MCP execution or production deployment validation.


In [ ]:
from pathlib import Path
import subprocess, sys

REPO = "https://github.com/msellamiTN/taas-provenance-discovery.git"
PIN = "410380107ac0e60bb634d1e79268032cd62b9421"
WORKDIR = Path("/content/taas-provenance-discovery")
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin"], check=True)
else:
    subprocess.run(["git", "clone", REPO, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--detach", PIN], check=True)
actual = subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True).strip()
assert actual == PIN, (actual, PIN)
print("Pinned revision:", actual)
%cd /content/taas-provenance-discovery


In [ ]:
!pip -q install -e '.[test,studio]'
!python -m compileall -q src/taas


In [ ]:
!pytest -q tests/test_retrieval_degenerate.py tests/test_studio_hypergraph_explorer.py tests/test_studio_navigation_shell.py tests/test_studio_professional_ux.py tests/test_hypergraph_projection.py tests/test_engine_certificate_gate.py


## Expected validation boundary

- One-document and duplicate-document LSA corpora produce finite normalized vectors without the zero-variance SVD warning.
- Non-degenerate corpora continue to use truncated SVD.
- The Studio homepage exposes the Hypergraph Explorer mount points.
- Missing bearer credentials are rejected; authenticated discovery returns a server-generated Core projection.
- The API integration test uses the real DiscoveryEngine and a local synthetic contract; it asserts `execution_performed=false`.

No MCP server is contacted and no tool is invoked. These tests use FastAPI TestClient; browser rendering, visual/keyboard behavior, remote deployment and production identity remain separate acceptance tests.
